
# taxonomy_master_cleanup

In [ ]:
#Problem: taxonomy_master_updated_old.csv contains invalid keys that appear to
# be category_group label values (e.g. RESTAURANTS__CAFES, TAKEAWAY___SNACKS)
# that leaked into the base_category_key column during file construction.
#
# Fix: cross-validate against cat_keys.csv (authoritative valid key list) and
# drop any row whose base_category_key is not a valid production key.
# Saves clean version as taxonomy_master_updated2.csv.
#
# Also diagnoses Run 3 results for rows affected by the invalid keys.

import pandas as pd
from pathlib import Path

SOURCE_PATH   = Path("assets/taxonomy_master_updated_old.csv")
OUTPUT_PATH   = Path("assets/taxonomy_master_updated2.csv")
CAT_KEYS_PATH = Path("assets/cat_keys.csv")
RUN3_PATH     = Path("outputs/run3_results.csv")  # update if filename differs


In [ ]:
# ============================================================
# PART 1 - Clean taxonomy_master_updated
# ============================================================

# --- 1. Load files ---
taxonomy = pd.read_csv(SOURCE_PATH)
cat_keys = pd.read_csv(CAT_KEYS_PATH)

print("=" * 55)
print("PART 1 - Taxonomy cleanup")
print("=" * 55)
print(f"Taxonomy rows (before): {len(taxonomy)}")

# --- 2. Identify the valid key column in cat_keys.csv ---
# Column was renamed to base_category_key in the CSV directly (09 Mar session)
# but handle both names defensively
if "base_category_key" in cat_keys.columns:
    valid_key_col = "base_category_key"
elif "key" in cat_keys.columns:
    valid_key_col = "key"
else:
    raise ValueError(
        f"Cannot find valid key column in cat_keys.csv. "
        f"Available: {cat_keys.columns.tolist()}"
    )

valid_keys = set(cat_keys[valid_key_col].str.strip().str.upper().dropna())
print(f"Valid keys from cat_keys.csv [{valid_key_col}]: {len(valid_keys)}")

# --- 3. Identify invalid rows ---
taxonomy["base_category_key"] = taxonomy["base_category_key"].str.strip().str.upper()
mask_invalid = ~taxonomy["base_category_key"].isin(valid_keys)
invalid_rows = taxonomy[mask_invalid]

print(f"\nInvalid keys found ({len(invalid_rows)} rows):")
for key in sorted(invalid_rows["base_category_key"].tolist()):
    suffix = "  <-- group label leakage" if "__" in key else ""
    print(f"  {key}{suffix}")

# --- 4. Save clean version ---
taxonomy_clean = taxonomy[~mask_invalid].copy()
taxonomy_clean.to_csv(OUTPUT_PATH, index=False)

print(f"\nRows removed:          {len(invalid_rows)}")
print(f"Taxonomy rows (after): {len(taxonomy_clean)}")
print(f"Saved to: {OUTPUT_PATH}")

# --- 5. Sanity check ---
remaining_invalid = taxonomy_clean[~taxonomy_clean["base_category_key"].isin(valid_keys)]
if remaining_invalid.empty:
    print("Sanity check PASSED - all remaining keys are valid")
else:
    print(f"Sanity check FAILED - {len(remaining_invalid)} invalid keys remain:")
    print(remaining_invalid["base_category_key"].tolist())

# --- 6. Valid keys missing from taxonomy (gaps) ---
missing_from_taxonomy = valid_keys - set(taxonomy_clean["base_category_key"])
if missing_from_taxonomy:
    print(f"\nValid keys absent from taxonomy ({len(missing_from_taxonomy)}):")
    for k in sorted(missing_from_taxonomy):
        print(f"  {k}")